# Activity Distance LFP: Cached Front-Crossing Coherence

This notebook estimates distance-dependent LFP excitability-front coherence from cached wavefront crossing results. It is intentionally cache-only: it reads `wavefront_*` HDF5 files and does not reload raw LFP, refilter signals, or recompute Hilbert phases.

In [ ]:
%matplotlib inline

import json
import os
import sys
import hashlib
import pickle
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault("MPLCONFIGDIR", str((repo_root / ".mpl-cache").resolve()))

from phase_waves import read_wavefront_cache

plt.rcParams["figure.dpi"] = 130
pd.set_option("display.max_columns", 120)
np.random.seed(0)


## Configuration

In [ ]:
DIV = 12
AGGREGATE_WELLS = [0, 1, 2, 3, 4, 5]
CACHE_PROFILE = "confirm"  # "confirm" by default; use "permissive" for the older fast screen.
LFP_SOURCE = "stim_removal_null_lfp"
WAVE_INTERVAL_MODE = "burst"
BAND_LOW = 30.0
BAND_HIGH = 50.0
FREQUENCY_HZ = 0.5 * (BAND_LOW + BAND_HIGH)
FS_DS = 500.0
LAMBDA_MIN = 0.5
LAMBDA_MAX = 10.0
FIT_RADIAL = True

MIN_DISTANCE_UM = 50.0
MAX_DISTANCE_UM = 3500.0
DISTANCE_BIN_UM = 100.0
MAX_REF_ELECTRODES_PER_EVENT = 150
MAX_TARGET_ELECTRODES_PER_EVENT = None
BOOTSTRAP_REPS = 1000
BOOTSTRAP_SEED = 0
FORCE_RECOMPUTE = False
STORE_PAIR_SAMPLES = False  # Pair-level tables are very large; event-bin summaries are enough for fast iteration.

OUTPUT_DIR = repo_root / "outputs" / "lfp_coherence_distance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WAVE_CACHE_DIR = repo_root / "outputs" / "lfp_waves"

CACHE_NOTE = f"{LFP_SOURCE}; {CACHE_PROFILE}; DIV{DIV}; {BAND_LOW:g}-{BAND_HIGH:g} Hz; cached wavefront crossings"


## Cache Paths and Loaders

In [ ]:
def dataset_for_well(well):
    return f"data{int(well):04d}"


def wavefront_cache_path(well, *, profile=CACHE_PROFILE):
    label = (
        f"{LFP_SOURCE}_{profile}_well{int(well)}_DIV{int(DIV)}_{dataset_for_well(well)}_{WAVE_INTERVAL_MODE}_"
        f"{BAND_LOW:g}-{BAND_HIGH:g}Hz_fs{FS_DS:g}_lambda{LAMBDA_MIN:g}-{LAMBDA_MAX:g}_radial{int(FIT_RADIAL)}"
    )
    return WAVE_CACHE_DIR / f"wavefront_{label}.h5"


def _as_dataframe(payload, *, well, path, group_name):
    columns = {key: value for key, value in payload.items() if isinstance(value, np.ndarray) and np.asarray(value).ndim > 0}
    if not columns:
        return pd.DataFrame()
    lengths = {key: len(value) for key, value in columns.items()}
    n = next(iter(lengths.values()))
    keep = {key: np.asarray(value) for key, value in columns.items() if len(value) == n}
    frame = pd.DataFrame(keep)
    frame.insert(0, "well", int(well))
    frame.insert(1, "cache_path", str(path))
    frame.insert(2, "group_name", group_name)
    return frame


def load_wavefront_tables(wells, *, profile=CACHE_PROFILE):
    event_parts = []
    local_parts = []
    summary_rows = []
    missing = []
    for well in wells:
        path = wavefront_cache_path(well, profile=profile)
        if not path.exists():
            fallback = wavefront_cache_path(well, profile="permissive")
            if profile != "permissive" and fallback.exists():
                print(f"well {well}: missing {profile} cache; using permissive fallback")
                path = fallback
            else:
                missing.append({"well": int(well), "path": str(path), "reason": "missing cache"})
                continue
        cache = read_wavefront_cache(path)
        if "wavefront_events" not in cache or "wavefront_local" not in cache:
            missing.append({"well": int(well), "path": str(path), "reason": "missing groups"})
            continue
        event = _as_dataframe(cache["wavefront_events"], well=well, path=path, group_name="wavefront_events")
        local = _as_dataframe(cache["wavefront_local"], well=well, path=path, group_name="wavefront_local")
        if local.empty:
            missing.append({"well": int(well), "path": str(path), "reason": "empty local table"})
            continue
        if "valid" in local:
            local["valid"] = local["valid"].astype(bool)
        if "front_valid" in event:
            event["front_valid"] = event["front_valid"].astype(bool)
        event_parts.append(event)
        local_parts.append(local)
        cfg = {}
        if "wavefront_calibration" in cache and cache["wavefront_calibration"].get("config"):
            cfg = json.loads(cache["wavefront_calibration"].get("config"))
        summary_rows.append({
            "well": int(well),
            "cache_path": str(path),
            "profile": cfg.get("profile", profile),
            "n_events": int(event["event_idx"].nunique()) if "event_idx" in event else int(len(event)),
            "n_local_rows": int(len(local)),
            "n_finite_crossings": int(np.isfinite(local["arrival_time_s"]).sum()),
            "n_valid_sources": int(local["valid"].sum()) if "valid" in local else 0,
        })
    event_df = pd.concat(event_parts, ignore_index=True) if event_parts else pd.DataFrame()
    local_df = pd.concat(local_parts, ignore_index=True) if local_parts else pd.DataFrame()
    cache_summary = pd.DataFrame(summary_rows)
    missing_df = pd.DataFrame(missing)
    return event_df, local_df, cache_summary, missing_df


event_df, local_df, cache_summary, missing_cache_df = load_wavefront_tables(AGGREGATE_WELLS)
display(cache_summary)
if not missing_cache_df.empty:
    display(missing_cache_df)
print(f"Loaded events={len(event_df):,}; local rows={len(local_df):,}; wells={cache_summary['well'].nunique() if not cache_summary.empty else 0}")


## Distance-Binned Crossing Coherence

In [ ]:
def distance_edges_and_centers():
    edges = np.arange(MIN_DISTANCE_UM, MAX_DISTANCE_UM + DISTANCE_BIN_UM, DISTANCE_BIN_UM, dtype=float)
    if edges[-1] < MAX_DISTANCE_UM:
        edges = np.append(edges, MAX_DISTANCE_UM)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return edges, centers


def analysis_cache_key():
    payload = {
        "cache_paths": sorted(cache_summary["cache_path"].tolist()) if not cache_summary.empty else [],
        "profile": CACHE_PROFILE,
        "frequency_hz": float(FREQUENCY_HZ),
        "distance": [float(MIN_DISTANCE_UM), float(MAX_DISTANCE_UM), float(DISTANCE_BIN_UM)],
        "max_ref": None if MAX_REF_ELECTRODES_PER_EVENT is None else int(MAX_REF_ELECTRODES_PER_EVENT),
        "max_target": None if MAX_TARGET_ELECTRODES_PER_EVENT is None else int(MAX_TARGET_ELECTRODES_PER_EVENT),
        "store_pair_samples": bool(STORE_PAIR_SAMPLES),
    }
    text = json.dumps(payload, sort_keys=True)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


ANALYSIS_CACHE_PATH = OUTPUT_DIR / f"lfp_crossing_coherence_{analysis_cache_key()}.pkl"
ANALYSIS_CACHE_PATH


In [ ]:
def _deterministic_subset(indices, max_n):
    indices = np.asarray(indices, dtype=int)
    if max_n is None or indices.size <= int(max_n):
        return indices
    pick = np.round(np.linspace(0, indices.size - 1, int(max_n))).astype(int)
    return indices[pick]


def compute_event_distance_coherence(local_df):
    edges, centers = distance_edges_and_centers()
    rows = []
    pair_rows = []
    if local_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    for (well, event_idx), group in local_df.groupby(["well", "event_idx"], sort=True):
        group = group.reset_index(drop=True)
        coords = group[["x_mm", "y_mm"]].to_numpy(float)
        arrival = group["arrival_time_s"].to_numpy(float)
        electrodes = group["electrode"].to_numpy(int) if "electrode" in group else np.arange(len(group), dtype=int)
        finite_target = np.isfinite(arrival) & np.all(np.isfinite(coords), axis=1)
        source_mask = finite_target & group["valid"].to_numpy(bool)
        source_idx = _deterministic_subset(np.flatnonzero(source_mask), MAX_REF_ELECTRODES_PER_EVENT)
        target_idx_all = _deterministic_subset(np.flatnonzero(finite_target), MAX_TARGET_ELECTRODES_PER_EVENT)
        if source_idx.size == 0 or target_idx_all.size < 2:
            continue
        dx = coords[target_idx_all, 0][None, :] - coords[source_idx, 0][:, None]
        dy = coords[target_idx_all, 1][None, :] - coords[source_idx, 1][:, None]
        distance_um = 1000.0 * np.sqrt(dx * dx + dy * dy)
        dt = arrival[target_idx_all][None, :] - arrival[source_idx][:, None]
        ref_e = electrodes[source_idx][:, None]
        target_e = electrodes[target_idx_all][None, :]
        not_self = ref_e != target_e
        bin_idx = np.searchsorted(edges, distance_um, side="right") - 1
        keep = not_self & (bin_idx >= 0) & (bin_idx < centers.size) & np.isfinite(dt)
        if not np.any(keep):
            continue
        phasor = np.exp(1j * 2.0 * np.pi * float(FREQUENCY_HZ) * dt[keep])
        kept_bin = bin_idx[keep].astype(int)
        kept_dist = distance_um[keep].astype(float)
        if STORE_PAIR_SAMPLES:
            pair_rows.append(pd.DataFrame({
                "well": int(well),
                "event_idx": int(event_idx),
                "distance_um": kept_dist,
                "distance_bin_idx": kept_bin,
                "distance_bin_center_um": centers[kept_bin],
                "phasor_real": np.real(phasor),
                "phasor_imag": np.imag(phasor),
            }))
        for b in np.unique(kept_bin):
            vals = phasor[kept_bin == b]
            if vals.size == 0:
                continue
            mean_phasor = np.mean(vals)
            rows.append({
                "well": int(well),
                "event_idx": int(event_idx),
                "distance_bin_idx": int(b),
                "distance_bin_center_um": float(centers[int(b)]),
                "coherence": float(np.abs(mean_phasor)),
                "mean_phase_rad": float(np.angle(mean_phasor)),
                "n_pairs": int(vals.size),
                "n_sources": int(source_idx.size),
                "n_targets": int(target_idx_all.size),
            })
    event_distance = pd.DataFrame(rows)
    pair_samples = pd.concat(pair_rows, ignore_index=True) if pair_rows else pd.DataFrame()
    return event_distance, pair_samples


if ANALYSIS_CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    with ANALYSIS_CACHE_PATH.open("rb") as f:
        cached = pickle.load(f)
    event_distance_coherence = cached["event_distance_coherence"]
    lfp_crossing_pair_samples = cached["lfp_crossing_pair_samples"]
    print(f"Loaded cached LFP distance coherence: {ANALYSIS_CACHE_PATH}")
else:
    event_distance_coherence, lfp_crossing_pair_samples = compute_event_distance_coherence(local_df)
    with ANALYSIS_CACHE_PATH.open("wb") as f:
        pickle.dump({
            "event_distance_coherence": event_distance_coherence,
            "lfp_crossing_pair_samples": lfp_crossing_pair_samples,
            "cache_summary": cache_summary,
        }, f)
    print(f"Wrote LFP distance coherence cache: {ANALYSIS_CACHE_PATH}")

print("event_distance_coherence", event_distance_coherence.shape)
print("lfp_crossing_pair_samples", lfp_crossing_pair_samples.shape)
event_distance_coherence.head()


## Bootstrap Summaries

In [ ]:
def bootstrap_ci(values, reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(int(seed))
    boot = np.array([np.nanmedian(rng.choice(values, size=values.size, replace=True)) for _ in range(int(reps))])
    return tuple(np.nanpercentile(boot, [2.5, 50.0, 97.5]))


def summarize_per_well(event_distance):
    rows = []
    for (well, distance), group in event_distance.groupby(["well", "distance_bin_center_um"], sort=True):
        lo, med, hi = bootstrap_ci(group["coherence"].to_numpy(float), seed=BOOTSTRAP_SEED + int(well))
        rows.append({
            "well": int(well),
            "distance_bin_center_um": float(distance),
            "coherence_median": med,
            "coherence_ci_low": lo,
            "coherence_ci_high": hi,
            "n_events": int(group["event_idx"].nunique()),
            "n_pairs_total": int(group["n_pairs"].sum()),
        })
    return pd.DataFrame(rows)


def hierarchical_aggregate_summary(event_distance):
    rows = []
    rng = np.random.default_rng(int(BOOTSTRAP_SEED) + 1000)
    wells = np.asarray(sorted(event_distance["well"].unique()), dtype=int)
    for distance, dist_group in event_distance.groupby("distance_bin_center_um", sort=True):
        observed = dist_group.groupby("well")["coherence"].median().median()
        boot = []
        for _ in range(int(BOOTSTRAP_REPS)):
            sampled_wells = rng.choice(wells, size=wells.size, replace=True)
            sampled_values = []
            for well in sampled_wells:
                values = dist_group.loc[dist_group["well"] == well, "coherence"].to_numpy(float)
                values = values[np.isfinite(values)]
                if values.size:
                    sampled_values.append(np.nanmedian(rng.choice(values, size=values.size, replace=True)))
            if sampled_values:
                boot.append(np.nanmedian(sampled_values))
        lo, med, hi = np.nanpercentile(boot, [2.5, 50.0, 97.5]) if boot else (np.nan, np.nan, np.nan)
        rows.append({
            "distance_bin_center_um": float(distance),
            "coherence_median": float(observed),
            "coherence_bootstrap_median": float(med),
            "coherence_ci_low": float(lo),
            "coherence_ci_high": float(hi),
            "n_wells": int(dist_group["well"].nunique()),
            "n_events": int(dist_group[["well", "event_idx"]].drop_duplicates().shape[0]),
            "n_pairs_total": int(dist_group["n_pairs"].sum()),
        })
    return pd.DataFrame(rows)


well_distance_summary = summarize_per_well(event_distance_coherence)
aggregate_distance_summary = hierarchical_aggregate_summary(event_distance_coherence)

display(well_distance_summary.head())
display(aggregate_distance_summary.head())


## Distance-Coherence Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

ax = axes[0, 0]
for well, group in well_distance_summary.groupby("well"):
    ax.plot(group["distance_bin_center_um"], group["coherence_median"], lw=1.5, label=f"well {well}")
    ax.fill_between(group["distance_bin_center_um"], group["coherence_ci_low"], group["coherence_ci_high"], alpha=0.10)
ax.set(title="LFP crossing coherence over distance by well", xlabel="distance (um)", ylabel="coherence")
ax.set_ylim(0, 1)
ax.legend(ncol=2)

ax = axes[0, 1]
ax.plot(aggregate_distance_summary["distance_bin_center_um"], aggregate_distance_summary["coherence_median"], color="black", lw=2.0, label="hierarchical median")
ax.fill_between(aggregate_distance_summary["distance_bin_center_um"], aggregate_distance_summary["coherence_ci_low"], aggregate_distance_summary["coherence_ci_high"], color="0.2", alpha=0.18, label="95% CI")
ax.set(title="Aggregate LFP crossing coherence", xlabel="distance (um)", ylabel="coherence")
ax.set_ylim(0, 1)
ax.legend()

pivot = well_distance_summary.pivot(index="well", columns="distance_bin_center_um", values="coherence_median")
im = axes[1, 0].imshow(pivot.to_numpy(), aspect="auto", origin="lower", vmin=0, vmax=1, extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min() - 0.5, pivot.index.max() + 0.5])
axes[1, 0].set(title="Per-well coherence heatmap", xlabel="distance (um)", ylabel="well")
fig.colorbar(im, ax=axes[1, 0], label="coherence")

support = event_distance_coherence.groupby("distance_bin_center_um", as_index=False).agg(n_events=("event_idx", "size"), n_pairs_total=("n_pairs", "sum"))
axes[1, 1].plot(support["distance_bin_center_um"], support["n_events"], label="event-bin rows")
axes[1, 1].set(title="Distance-bin support", xlabel="distance (um)", ylabel="event-bin rows")
axes_support = axes[1, 1].twinx()
axes_support.plot(support["distance_bin_center_um"], support["n_pairs_total"], color="tab:orange", label="pairs")
axes_support.set_ylabel("pairs")
axes[1, 1].legend(loc="upper left")
axes_support.legend(loc="upper right")

fig.suptitle(CACHE_NOTE)
plt.show()


## Save Tables

In [ ]:
table_dir = OUTPUT_DIR / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
cache_summary.to_csv(table_dir / "cache_summary.csv", index=False)
event_df.to_csv(table_dir / "wavefront_events.csv", index=False)
# local_df is large; save only compact rows needed for this notebook.
local_df[["well", "event_idx", "electrode", "x_mm", "y_mm", "arrival_time_s", "valid"]].to_csv(table_dir / "wavefront_local_compact.csv", index=False)
if STORE_PAIR_SAMPLES and not lfp_crossing_pair_samples.empty:
    lfp_crossing_pair_samples.to_csv(table_dir / "lfp_crossing_pair_samples.csv", index=False)
event_distance_coherence.to_csv(table_dir / "event_distance_coherence.csv", index=False)
well_distance_summary.to_csv(table_dir / "well_distance_summary.csv", index=False)
aggregate_distance_summary.to_csv(table_dir / "aggregate_distance_summary.csv", index=False)
print(f"Wrote tables to {table_dir}")
